- VLM 모델 학습을 위해 COCO 데이터셋 처럼 하나의 이미지 당 5개의 문장 라벨링
- Gemini 2.5 Flash API를 활용한 라벨링

In [23]:
import google.generativeai as genai
import json
import os
from PIL import Image
import shutil
from dotenv import load_dotenv
import requests
import base64

In [9]:

# 원본 이미지 경로: 이전 프로젝트 경로에 있음
IMG_SRC = r"C:\Users\ASUS\Desktop\제로베이스\딥러닝 프로젝트\낙상사고 위험동작 영상-센서 쌍 데이터_병원,후면낙상\3.개방데이터\1.데이터\Training\01.원천데이터\TS\이미지"

# 원본 라벨 경로: 현재 프로젝트 경로에 있음
PROJECT = r"C:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM"
LBL_SRC = os.path.join(PROJECT, "labels", "이미지")

# 임시 폴더
TMP_IMG = os.path.join(PROJECT, "tmp_test", "images")
TMP_LBL = os.path.join(PROJECT, "tmp_test", "labels_original")
TMP_OUT = os.path.join(PROJECT, "tmp_test", "test_labels")

# 기존 정리 후 재생성
if os.path.exists(os.path.join(PROJECT, "tmp_test")):
    shutil.rmtree(os.path.join(PROJECT, "tmp_test"))

os.makedirs(TMP_IMG, exist_ok=True)
os.makedirs(TMP_LBL, exist_ok=True)
os.makedirs(TMP_OUT, exist_ok=True)

# ★ 1개 시나리오의 이미지 10장 전부 복사
scenario = "00127_H_A_SY_C3"

img_folder = os.path.join(IMG_SRC, "Y", "SY", scenario)
lbl_folder = os.path.join(LBL_SRC, "Y", "SY", scenario)
for f in sorted(os.listdir(img_folder)):
    if not f.lower().endswith('.jpg'):
        continue
    
    # 이미지 복사
    shutil.copy2(os.path.join(img_folder, f), os.path.join(TMP_IMG, f))
    
    # 대응 라벨 복사
    json_name = f.replace(".jpg", ".json").replace(".JPG", ".json")
    lbl_path = os.path.join(lbl_folder, json_name)
    if os.path.exists(lbl_path):
        shutil.copy2(lbl_path, os.path.join(TMP_LBL, json_name))
print(f"시나리오: {scenario}")
print(f"이미지 {len(os.listdir(TMP_IMG))}장: {sorted(os.listdir(TMP_IMG))}")
print(f"라벨 {len(os.listdir(TMP_LBL))}개: {sorted(os.listdir(TMP_LBL))}")

시나리오: 00127_H_A_SY_C3
이미지 10장: ['00127_H_A_SY_C3_I001.JPG', '00127_H_A_SY_C3_I002.JPG', '00127_H_A_SY_C3_I003.JPG', '00127_H_A_SY_C3_I004.JPG', '00127_H_A_SY_C3_I005.JPG', '00127_H_A_SY_C3_I006.JPG', '00127_H_A_SY_C3_I007.JPG', '00127_H_A_SY_C3_I008.JPG', '00127_H_A_SY_C3_I009.JPG', '00127_H_A_SY_C3_I010.JPG']
라벨 10개: ['00127_H_A_SY_C3_I001.json', '00127_H_A_SY_C3_I002.json', '00127_H_A_SY_C3_I003.json', '00127_H_A_SY_C3_I004.json', '00127_H_A_SY_C3_I005.json', '00127_H_A_SY_C3_I006.json', '00127_H_A_SY_C3_I007.json', '00127_H_A_SY_C3_I008.json', '00127_H_A_SY_C3_I009.json', '00127_H_A_SY_C3_I010.json']


In [24]:
# .env 파일 로드
load_dotenv("API_KEY_to_Labelling.env")
API_KEY = os.getenv("GOOGLE_API_KEY")
URL = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-preview-05-20:generateContent?key={API_KEY}"

In [25]:
PROMPT = """당신은 병원 환자 안전 모니터링 전문가입니다.
이 이미지를 분석하여 환자 상태를 보고하세요.

## 판단 기준 (우선순위 순)
1. [fall] 환자의 몸이 바닥에 접촉 -> "낙상 발생"
2. [bed_exit] 침대가 보이고 + 환자가 침대를 이탈하는 동작 -> "침대 이탈"
3. [moving] 환자가 서서 걷는 동작 -> "이동 중"
4. [resting] 환자가 침대에 누워있음 -> "휴식 중"

## 출력 형식 (반드시 JSON으로)
{
 "status": "fall | bed_exit | moving | resting",
 "captions": [
 "한국어 상태 설명 1 (10 ~ 20단어, 의료 보고서 문체)",
 "한국어 상태 설명 2 (같은 상황을 다른 표현으로)",
 "한국어 상태 설명 3 (자세/위치 중심 묘사)",
 "한국어 상태 설명 4 (위험도 중심 묘사)",
 "한국어 상태 설명 5 (간결한 한 줄 보고)"
 ]
}

JSON만 출력하세요.
"""

In [27]:
# 제미나이 API 호출
def call_gemini(image_path, prompt):
    """Gemini 2.5 Flash API 호출 (REST)"""
    with open(image_path, "rb") as f:
        img_b64 = base64.b64encode(f.read()).decode("utf-8")
    
    payload = {
        "contents": [{
            "parts": [
                {"text": prompt},
                {"inline_data": {"mime_type": "image/jpeg", "data": img_b64}}
            ]
        }]
    }
    
    response = requests.post(URL, json=payload)
    result = response.json()
    
    # 응답 텍스트 추출
    text = result["candidates"][0]["content"]["parts"][0]["text"].strip()
    
    # ```json ... ``` 제거
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    
    return json.loads(text)

print("API 설정 완료!")
print(f"API Key: {API_KEY[:10]}..." if API_KEY else "⚠️ API Key 없음!")

API 설정 완료!
API Key: AIzaSyCgNX...


In [28]:
PROJECT = r"C:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM"
TMP_IMG = os.path.join(PROJECT, "tmp_test", "images")
TMP_LBL = os.path.join(PROJECT, "tmp_test", "labels_original")
TMP_OUT = os.path.join(PROJECT, "tmp_test", "test_labels")

for img_name in sorted(os.listdir(TMP_IMG)):
    if not img_name.lower().endswith('.jpg'):
        continue
    
    json_name = img_name.replace(".jpg", ".json").replace(".JPG", ".json")
    label_path = os.path.join(TMP_LBL, json_name)
    
    # 1) 기존 JSON 읽기
    if not os.path.exists(label_path):
        print(f"[SKIP] 라벨 없음: {json_name}")
        continue
    
    with open(label_path, 'r', encoding='utf-8') as f:
        original_data = json.load(f)
    
    print(f"\n처리 중: {img_name}")
    
    # 2) Gemini API 호출
    try:
        vlm_result = call_gemini(os.path.join(TMP_IMG, img_name), PROMPT)
        
        # 3) 기존 JSON에 vlm_labels 병합
        original_data["vlm_labels"] = {
            "status": vlm_result["status"],
            "captions": vlm_result["captions"]
        }
        
        # 4) test_labels에 저장
        save_path = os.path.join(TMP_OUT, json_name)
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(original_data, f, ensure_ascii=False, indent=2)
        
        print(f"  상태: {vlm_result['status']}")
        for i, cap in enumerate(vlm_result['captions'], 1):
            print(f"  캡션{i}: {cap}")
            
    except Exception as e:
        print(f"  [ERROR] {e}")

print(f"\n완료! 결과: {sorted(os.listdir(TMP_OUT))}")


처리 중: 00127_H_A_SY_C3_I001.JPG
  [ERROR] 'candidates'

처리 중: 00127_H_A_SY_C3_I002.JPG
  [ERROR] 'candidates'

처리 중: 00127_H_A_SY_C3_I003.JPG
  [ERROR] 'candidates'

처리 중: 00127_H_A_SY_C3_I004.JPG
  [ERROR] 'candidates'

처리 중: 00127_H_A_SY_C3_I005.JPG
  [ERROR] 'candidates'

처리 중: 00127_H_A_SY_C3_I006.JPG
  [ERROR] 'candidates'

처리 중: 00127_H_A_SY_C3_I007.JPG
  [ERROR] 'candidates'

처리 중: 00127_H_A_SY_C3_I008.JPG
  [ERROR] 'candidates'

처리 중: 00127_H_A_SY_C3_I009.JPG
  [ERROR] 'candidates'

처리 중: 00127_H_A_SY_C3_I010.JPG
  [ERROR] 'candidates'

완료! 결과: []


In [ ]:
for f in sorted(os.listdir(TMP_OUT)):
    with open(os.path.join(TMP_OUT, f), 'r', encoding='utf-8') as fp:
        data = json.load(fp)
    
    print(f"\n{'='*50}")
    print(f"파일: {f}")
    print(f"상태: {data['vlm_labels']['status']}")
    for i, cap in enumerate(data['vlm_labels']['captions'], 1):
        print(f"  {i}. {cap}")
